# Milestone 01 — ResNet18 Image Embeddings

**Status:** A reproducible feature-extraction demo, **not** Nepali caption generation.

**Research rationale:** Before adapting a visual encoder to a Reformer language decoder, check whether the encoder produces stable 512-dimensional image representations. We compare the first image with its mirrored version and an unrelated image; similarity is descriptive and is *not* a caption-quality benchmark.

Run this notebook from the repository root. The first cell downloads the ~45 MB pretrained checkpoint if not cached.

In [ ]:
from pathlib import Path
import sys
import torch
from PIL import Image, ImageOps
ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.encoder import build_encoder, encode
A = Image.open(ROOT / "assets/astronaut.png").convert("RGB")
B = Image.open(ROOT / "assets/chelsea.png").convert("RGB")
print("Image A:", A.size, "Image B:", B.size)

In [ ]:
# PRETRAINED mode. Do not replace this with random weights when reporting scientific observations.
model, preprocess = build_encoder("pretrained", device="cpu")
emb_a = encode(A, model, preprocess)
emb_a_repeat = encode(A, model, preprocess)
emb_mirror = encode(ImageOps.mirror(A), model, preprocess)
emb_b = encode(B, model, preprocess)
print("Embedding shape:", tuple(emb_a.shape))
print("First 8 values:", emb_a[:8].tolist())
print("Repeated inference equal:", torch.equal(emb_a, emb_a_repeat))

In [ ]:
cos = lambda a,b: torch.nn.functional.cosine_similarity(a[None,:], b[None,:]).item()
print("Cosine - same photo:", cos(emb_a, emb_a_repeat))
print("Cosine - mirrored photo:", cos(emb_a, emb_mirror))
print("Cosine - other photo:", cos(emb_a, emb_b))
print("Interpretation: These are visual features, not captions or accuracy metrics.")

## Interpretation and caveats

- A 512-D feature shape verifies that the final classifier head has been removed.
- The repeated image should give the same output in evaluation mode.
- No claims about Nepali semantics, caption generation, or Reformer efficiency follow from these numbers.
- To produce an auditable JSON report and save embeddings, run `python -m src.run_experiment --weights pretrained`.

**Next:** Repeat on an approved Nepali image-caption dataset and investigate how visual embeddings should condition a language model.